# 01 — Data Understanding
Inspect the nine Olist source tables, their grains, keys, nulls, and date coverage before analysis.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
RAW

In [ ]:
files = sorted(RAW.glob('*.csv'))
profile_rows = []
tables = {}
for path in files:
    data = pd.read_csv(path, low_memory=False)
    tables[path.stem] = data
    profile_rows.append({
        'file': path.name, 'rows': len(data), 'columns': len(data.columns),
        'duplicate_rows': int(data.duplicated().sum()),
        'null_cells': int(data.isna().sum().sum()),
    })
pd.DataFrame(profile_rows).sort_values('rows', ascending=False)

## Key identity check
`customer_id` joins orders to customers; `customer_unique_id` represents the durable customer for repeat behavior.

In [ ]:
customers = tables['olist_customers_dataset']
pd.Series({
    'customer_id_unique': customers['customer_id'].is_unique,
    'customer_unique_id_count': customers['customer_unique_id'].nunique(),
    'multi_identity_customers': int((customers['customer_unique_id'].value_counts() > 1).sum()),
})

In [ ]:
orders = tables['olist_orders_dataset'].copy()
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders.groupby(orders['order_purchase_timestamp'].dt.to_period('M')).size().tail(8)